### Popular pre-trained models for vision tasks
* VGG
    * VGG16/VGG19 : Deep Networks with 16 or 19 layers
    * Known for simplicity in architecture : Stack of convolutional layers followed by fully connected layers
    * Applications: General Purpose image , classification and feature extractions
* ResNet
    * Residual Networks: Introduced residual connections (skip connections) to tackle vanishing gradients
    * Popular Variants: ResNet18, ResNet50m ResNet, 101
    * Applications: Large Scale image classification tasks, object detection
    * Inception
        * InceptionV3: Known for inception modules , which allow for multi-scale feature extraction in one layer
        * Applications: Scene Recognition , fine Grained Image classification
* EfficientNet
    * Family of models that scales network depth, width and resolution efficiently
    * provides better performance with fewer parameters

### Freezing and Unfreezing Layers for fine tuning
* Why Freeze layers? 
    * Early Layers in pre-trained models captures general featreus
    * Freezing these layers reduces training time and prevents overfitting on small datasets
* Why Unfreeze Layers?
    * Later layers learn task specific feratures
    * Unfreezing layers allow the model to adapt to new task
* Approach:
    * Initial Training
        * Freeze most layers and train last few layers
    * Fine tuning
        * Gradually unfreeze layers and reduce the learning rate for fine-tuning

---

### Using Transfer Learning for image classification tasks
* Steps:
    * Load a  pre-trained model
    * Replace the last layer with a task specific classifier (eg, softmax for multi-class classification)
    * Fine- tune the model on the new dataset
    

In [11]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [2]:
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224,224,3))


E0000 00:00:1781719904.531022 2282350 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 32s 0us/step


In [3]:
#  Freeze all layers in base model
for layer in base_model.layers:
    layer.trainable = False

In [6]:
# Add Custom classification head
x = Flatten()(base_model.output)
x = Dense(256, activation="relu")(x)
output = Dense(5, activation='softmax')(x)


In [8]:
model = Model(inputs=base_model.input, outputs =output)



In [9]:
# compile the model
model.compile(optimizer="adam",loss="categorical_crossentropy", metrics=["accuracy"])

In [10]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 49,279,365 (187.99 MB)

 Trainable params: 25,691,653 (98.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
# Data Prep
data_gen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_data = data_gen.flow_from_directory("PATH_TO_DATASET",
                                          target_size=(224,224),
                                          batch_size=32,
                                          class_mode="categorical",subset="training")
val_data = data_gen.flow_from_directory("PATH_TO_DATASET",
                                          target_size=(224,224), batch_size=32,
                                          class_mode="categorical",subset="validation")

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    steps_per_epoch=len(train_data),
    validation_steps=len(val_data)
)

FileNotFoundError: [Errno 2] No such file or directory: 'PATH_TO_DATASET'

In [ ]:
for layer in base_model.layers[-5:]:
    layer.trainable=True

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),loss="categorical_crossentropy", metrics=['accuracy'])

val_loss , val_accuracy = model.evaluate(val_data)
